# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Skill loaded: `building-baselines` + `flyrank/flyrank-data`.*

**Lane (freestyle):** a combined refresh + CTR-fix review queue — pages that are either gone stale while still visible, or ranking fine but under-clicking for their position tier. This uses the starter CSV (`data/raw/content_refresh_anonymized.csv`), not the warehouse — the lane question doesn't need the full 79M-row panel, just the 30k-row teaching slice.

In [2]:
import pandas as pd
import numpy as np
import os

if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/moriyamao/flyrank-ml-internship.git repo
    os.chdir('repo')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df.shape)

Cloning into 'repo'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 145 (delta 52), reused 95 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 1.85 MiB | 13.34 MiB/s, done.
Resolving deltas: 100% (52/52), done.
(30000, 45)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** a page belongs in the review queue if it has real search visibility (≥300 impressions in 90 days) AND either (a) it has gone stale (91+ days since last update), or (b) it ranks well but its CTR sits below the median CTR for its own position tier.

**Signal check 1 — staleness behind the refresh flag** (`freshness_tier` vs `is_declining_label`, used here only to verify the signal, never as a rule input):

| freshness_tier | n | decline_rate |
|---|---|---|
| 0-30 | 20,480 | 0.511 |
| 31-90 | 175 | 0.589 |
| 91-180 | 9,171 | 0.611 |
| 181+ | 174 | 0.471 |

**Verdict: MIXED.** The core comparison holds — 91-180 days stale shows a materially higher decline rate (61.1%) than fresh 0-30 (51.1%), which is what the refresh flag assumes. But the pattern isn't monotonic: the 181+ tier drops back to 47.1%, and both 31-90 and 181+ have very small n (175 and 174 rows) against the two big tiers (20,480 and 9,171) — too small to trust on their own. I'm keeping staleness in the rule, but only using the two well-populated tiers (0-30 vs 91-180) as the justification, not the noisy extremes.

**Signal check 2 — CTR-vs-position behind the CTR-fix logic** (`position_tier` vs `ctr`):

| position_tier | n | avg_ctr |
|---|---|---|
| top_3 | 2,321 | 1.484 |
| page_1 | 11,814 | 0.652 |
| striking | 7,304 | 0.323 |
| page_3_5 | 7,242 | 0.222 |
| deep | 1,319 | 0.150 |

**Verdict: CONFIRMED.** CTR drops monotonically and sharply as position worsens (1.48% → 0.65% → 0.32% → 0.22% → 0.15%), exactly what the CTR-fix logic assumes. That makes "CTR below the median for your own position tier" a fair, apples-to-apples underperformance flag rather than just re-detecting bad position.

In [3]:
# Signal 1: staleness vs decline rate -- bucket table with n
signal1 = df.groupby('freshness_tier').agg(n=('content_id','size'), decline_rate=('is_declining_label','mean')).round(3)
signal1

,n,decline_rate
freshness_tier,,
0-30,20480,0.511
181+,174,0.471
31-90,175,0.589
91-180,9171,0.611


In [4]:
# Signal 2: position tier vs avg ctr -- bucket table with n
signal2 = df.groupby('position_tier').agg(n=('content_id','size'), avg_ctr=('ctr','mean')).round(3)
signal2

,n,avg_ctr
position_tier,,
deep,1319,0.150
page_1,11814,0.652
page_3_5,7242,0.222
striking,7304,0.323
top_3,2321,1.484


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import os

visible = df['impressions_90d'] >= 300
stale = df['freshness_tier'].isin(['91-180', '181+'])

# CTR gap: below the median CTR for the SAME position tier -- an apples-to-apples underperformance
# flag, not a re-statement of bad position. Only defined for tiers where ranking is already decent.
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
ctr_gap = (df['ctr'] < tier_median_ctr) & (df['position_tier'].isin(['top_3', 'page_1', 'striking']))

df['visible_flag'] = visible.astype(int)
df['stale_flag'] = stale.astype(int)
df['ctr_gap_flag'] = ctr_gap.astype(int)

# Transparent score: readable on purpose, no fitted weights.
df['score'] = df['visible_flag'] * (df['stale_flag'] + df['ctr_gap_flag']) * np.log1p(df['impressions_90d'])

def reason_code(row):
    if row['stale_flag'] and row['ctr_gap_flag']:
        return 'stale_and_ctr_gap'
    if row['stale_flag']:
        return 'stale_but_visible'
    if row['ctr_gap_flag']:
        return 'ctr_gap_for_position'
    return 'no_flag'

def action_label(row):
    if row['stale_flag'] and row['ctr_gap_flag']:
        return 'refresh_and_fix_ctr'
    if row['stale_flag']:
        return 'refresh'
    if row['ctr_gap_flag']:
        return 'fix_ctr'
    return 'monitor'

df['reason_code'] = df.apply(reason_code, axis=1)
df['action'] = df.apply(action_label, axis=1)

queue = df[df['score'] > 0].sort_values('score', ascending=False)
print('Queue size:', len(queue), 'of', len(df))
print(queue['reason_code'].value_counts())
print(queue['action'].value_counts())

os.makedirs('work/outputs', exist_ok=True)
out_cols = ['content_id', 'client_id', 'score', 'reason_code', 'action',
            'impressions_90d', 'ctr', 'avg_position', 'freshness_tier', 'position_tier']
queue[out_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print('Written: work/outputs/baseline_action_score.csv')

Queue size: 9894 of 30000
reason_code
stale_but_visible       5526
ctr_gap_for_position    2660
stale_and_ctr_gap       1708
Name: count, dtype: int64
action
refresh                5526
fix_ctr                2660
refresh_and_fix_ctr    1708
Name: count, dtype: int64
Written: work/outputs/baseline_action_score.csv


## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top10 = queue.head(10)[['content_id', 'client_id', 'score', 'reason_code', 'action',
                         'impressions_90d', 'ctr', 'avg_position', 'freshness_tier', 'position_tier']]
top10

,content_id,client_id,score,reason_code,action,impressions_90d,ctr,avg_position,freshness_tier,position_tier
6653,content_5fe46e04994d,client_4e07408562,26.314364,stale_and_ctr_gap,refresh_and_fix_ctr,517715,0.14,4.2,91-180,page_1
3394,content_36ff89c8214e,client_19581e27de,25.190126,stale_and_ctr_gap,refresh_and_fix_ctr,295097,0.05,7.3,91-180,page_1
7445,content_c8e9d6ab9013,client_19581e27de,24.497105,stale_and_ctr_gap,refresh_and_fix_ctr,208678,0.00,9.7,91-180,page_1
26474,content_a7427266c305,client_19581e27de,24.423234,stale_and_ctr_gap,refresh_and_fix_ctr,201111,0.11,5.7,91-180,page_1
3070,content_91652435f57a,client_19581e27de,23.960739,stale_and_ctr_gap,refresh_and_fix_ctr,159590,0.06,7.8,91-180,page_1
29868,content_f42eb861c6dd,client_19581e27de,23.869420,stale_and_ctr_gap,refresh_and_fix_ctr,152467,0.13,6.5,91-180,page_1
14190,content_11fcfd65d94c,client_19581e27de,23.824530,stale_and_ctr_gap,refresh_and_fix_ctr,149083,0.15,6.2,91-180,page_1
5621,content_97a86caf3a3d,client_19581e27de,23.805484,stale_and_ctr_gap,refresh_and_fix_ctr,147670,0.07,6.4,91-180,page_1
9193,content_c1fe78bc4e37,client_19581e27de,23.612026,stale_and_ctr_gap,refresh_and_fix_ctr,134055,0.03,7.5,91-180,page_1
2476,content_4c76e9b13aea,client_19581e27de,23.518837,stale_and_ctr_gap,refresh_and_fix_ctr,127952,0.07,7.4,91-180,page_1


**Top 10, one line each** (all ten landed as `stale_and_ctr_gap` → `refresh_and_fix_ctr`, the highest-score bucket since it satisfies both flags at once — real, not cherry-picked):

1. `content_5fe46e04994d` — refresh_and_fix_ctr, 517,715 impressions/90d but CTR 0.14% at position 4.2 while stale 91-180 days. Confidence: high, huge visibility makes the CTR gap costly. Would be wrong if the low CTR is a SERP feature (a featured snippet stealing clicks) rather than a fixable on-page issue.
2. `content_36ff89c8214e` — same pattern, CTR 0.05% at position 7.3. Confidence: high. Would be wrong if this keyword's intent is navigational and low CTR is expected (branded query cannibalized by another result).
3. `content_c8e9d6ab9013` — CTR literally 0.00% at position 9.7. Confidence: medium — zero CTR at real impression volume (208k) is suspicious enough to double-check tracking before assuming it's purely a content problem.
4. `content_a7427266c305` — CTR 0.11% at position 5.7, stale. Confidence: high. Would be wrong if `days_since_last_update` is stale due to a CMS migration timestamp reset, not genuine content staleness.
5. `content_91652435f57a` — CTR 0.06% at position 7.8, stale. Confidence: high. Would be wrong if the page recently lost a rich result (review stars, FAQ) that used to drive the clicks this baseline expects.
6. `content_f42eb861c6dd` — CTR 0.13% at position 6.5, stale. Confidence: high. Would be wrong if seasonal demand cratered independent of the page itself (the rule can't see demand shifts, only CTR).
7. `content_11fcfd65d94c` — CTR 0.15% at position 6.2, stale. Confidence: high, consistent with the pattern above.
8. `content_97a86caf3a3d` — CTR 0.07% at position 6.4, stale. Confidence: high. Would be wrong if this is a duplicate/near-duplicate of another ranked page splitting its own clicks.
9. `content_c1fe78bc4e37` — CTR 0.03% at position 7.5, stale. Confidence: medium — CTR this low for page-1 position is an outlier even within this list; worth a manual SERP check before refreshing.
10. `content_4c76e9b13aea` — CTR 0.07% at position 7.4, stale. Confidence: high, same shape as the rest of the list.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest pick: #3 (`content_c8e9d6ab9013`)** — a CTR of exactly 0.00% on 208,678 impressions is the one entry here that looks less like "content needs a refresh" and more like "something is broken" (tracking gap, redirect loop, or a SERP feature absorbing 100% of clicks). It scores near the top because the rule can't tell "content problem" from "measurement problem" — that's a real weakness of a CTR-only signal, not a data error, so it stays in the queue but gets a manual check first rather than an automatic refresh.

**Leakage check:**
- The rule inputs are `impressions_90d`, `freshness_tier`, `position_tier`, and `ctr` — all knowable *before* the moment of decision. None of them are downstream of `trend_direction` or `trend_pct`.
- `is_declining_label` was used **only** in Section 1 to verify the staleness signal held up — it never enters `score`, `reason_code`, or `action` in Section 2. Confirmed by inspecting the score formula: it references `visible_flag`, `stale_flag`, `ctr_gap_flag`, and `impressions_90d` only.
- No product-decision flags (`provider_used`, `model_used`) or future-window columns (`impressions_last_30d` vs `impressions_prev_30d` trend deltas) feed the rule.
- No client names, URLs, or private query text appear anywhere in this notebook — only pseudonymous IDs.

In [7]:
# Leakage check, verified in code rather than just asserted:
rule_inputs = {'visible_flag', 'stale_flag', 'ctr_gap_flag', 'impressions_90d'}
label_derived = {'trend_direction', 'trend_pct', 'is_declining_label'}
print('Rule inputs touch any label-derived column?', bool(rule_inputs & label_derived))

Rule inputs touch any label-derived column? False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run it yourself once in your own Colab/repo to confirm**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.